# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [39]:
%reload_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [41]:
# I selected Managing Oneself — Peter Drucker (Harvard Business Review) Source: PDF provided in the assignment.
!pip install langchain langchain-community pypdf
from langchain_community.document_loaders import PyPDFLoader

# URL provided in assignment
pdf_url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

# Load the document
loader = PyPDFLoader(pdf_url)
docs = loader.load()

# Check output structure
print(f"Total pages loaded: {len(docs)}")
print(docs[0].page_content[:500])

# Join all pages into a single string for easier processing

document_text = ""

for page in docs:
    document_text += page.page_content + "\n"

zsh:1: command not found: pip
Total pages loaded: 13
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
# pip install openai pydantic langchain-community pypdf
from __future__ import annotations

from typing import Optional
from pydantic import BaseModel, Field, ConfigDict

from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader

In [44]:
import os
from openai import OpenAI
GATEWAY_BASE_URL = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"
client = OpenAI(
    base_url=GATEWAY_BASE_URL,
    api_key="any value", 
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

In [ ]:
from pydantic import BaseModel, Field, ConfigDict

# Define the output schema as a Pydantic model
class ArticleStructuredOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# Developer instructions for the model to follow when generating the output
DEVELOPER_INSTRUCTIONS = """
You are an expert document summarizer for professionals.
You must:
- Follow the output schema exactly (no extra fields).
- Keep Relevance to a single paragraph.
- Write Summary in a clearly distinguishable tone specified by the user.
- Keep Summary concise and under 1000 tokens.
- Do not fabricate facts not supported by the provided document text.
"""

TONE = "Bureaucratese"

# The user prompt includes the article text and specific instructions for the output format and tone.
USER_PROMPT_TEMPLATE = """
Summarize the following article and return a structured output matching the schema.

Tone requirement:
- Use the tone: "{tone}"
- Make the tone obvious and consistent throughout the Summary (bureaucratic phrasing, policy-like cadence, formal administrative voice).

Context (the article text):
\"\"\"\n{context}\n\"\"\"

Notes:
- Title and Author must be extracted from the article.
- InputTokens and OutputTokens must be set to 0 in your generated object (placeholders).
  They will be overwritten programmatically from the API response usage fields.
"""

user_prompt = USER_PROMPT_TEMPLATE.format(tone=TONE, context=document_text)

# Use the OpenAI client to send the request with the developer instructions and user prompt, specifying the model to use.
MODEL = "gpt-4o-mini" 
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": DEVELOPER_INSTRUCTIONS},
        {"role": "user", "content": user_prompt},
    ],
    text_format=ArticleStructuredOutput,
)

out: ArticleStructuredOutput = response.output_parsed

# After receiving the response, set the InputTokens and OutputTokens fields based on the API response usage data
out.InputTokens = response.usage.input_tokens
out.OutputTokens = response.usage.output_tokens

print(out.model_dump_json(indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "The article emphasizes the importance of self-management in the modern knowledge economy, positing that individuals must actively manage their own careers and understand their strengths, weaknesses, values, and performance methods to achieve success and fulfillment.",
  "Summary": "In the contemporary knowledge economy, individuals are entrusted with the responsibility of managing their own careers, akin to being their own chief executive officers. A profound self-awareness encompassing one’s strengths, weaknesses, and preferred methods of working is paramount for achieving excellence. The article delineates key questions aiding self-discovery: identification of strengths via feedback analysis, understanding individual learning styles, reconciling personal values with organizational values, discerning optimal working environments, and determining one's contributions to the organization. Moreover, it unders

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
import asyncio
from typing import Optional
try:
    from deepeval.models.base_model import DeepEvalBaseLLM
except ImportError:
    from deepeval.models import DeepEvalBaseLLM 

In [ ]:
class GatewayOpenAILLM(DeepEvalBaseLLM):
    def __init__(self, client, model: str = "gpt-4o-mini", max_tokens: int = 700, temperature: float = 0):
        self._client = client
        self._model = model
        self._max_tokens = max_tokens
        self._temperature = temperature
        self._loaded = False

    def load_model(self):
        self._loaded = True
        return self

    def get_model_name(self) -> str:
        return self._model

    # Sync generate required by many metrics
    def generate(self, prompt: str) -> str:
        if not self._loaded:
            self.load_model()

        resp = self._client.responses.create(
            model=self._model,
            input=prompt,
            max_output_tokens=self._max_tokens,
            temperature=self._temperature,
        )
        return resp.output_text

    # Async generate required by your DeepEval version
    async def a_generate(self, prompt: str) -> str:
        return await asyncio.to_thread(self.generate, prompt)

In [64]:
def evaluate_summary_with_deepeval(
    document_text: str,
    summary_text: str,
    client, 
    judge_model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    # Develop DeepEval test case：input=original text，actual_output=summary
    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text,
    )

    evaluator = GatewayOpenAILLM(client=client, model=judge_model)

    # Build SummarizationMetric
    summarization_questions = [
        "Does the summary clearly state the article’s main purpose as managing oneself (self-management for knowledge workers)?",
        "Does the summary accurately describe 'feedback analysis' as comparing predicted outcomes with results later to identify strengths?",
        "Does the summary include the claim that performance is built on strengths rather than fixing weaknesses?",
        "Does the summary mention understanding one’s working style / learning style and matching roles accordingly?",
        "Does the summary mention aligning work with personal values and the consequences of mismatch?",
        "Does the summary cover taking responsibility for relationships (understanding others’ strengths/styles/values)?",
        "Does the summary mention planning for a long career, including a second half/parallel path?"
    ]

    summarization_metric = SummarizationMetric(
        model=evaluator,
        assessment_questions=summarization_questions,
        include_reason=True,
        threshold=0.5
    )

    # GEval: Coherence/Clarity（5 steps）
    coherence_steps = [
        "Check whether the summary has a clear structure (opening premise, key points, conclusion).",
        "Assess whether ideas flow logically with explicit transitions and minimal abrupt jumps.",
        "Identify ambiguity or underspecified references that reduce clarity.",
        "Check for redundancy; penalize repeated points that do not add new information.",
        "Assess readability: concise sentences, coherent paragraphs, and minimal jargon."
    ]
    coherence_metric = GEval(
        name="Coherence/Clarity",
        model=evaluator,
        evaluation_steps=coherence_steps,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )

    # GEval: Tonality（5 steps）
    tonality_steps = [
        "Determine whether the tone is consistently Bureaucratese (formal, administrative, policy-like) throughout.",
        "Check for stylistic markers of Bureaucratese (procedural phrasing, institutional voice, structured enumerations).",
        "Penalize colloquial language, slang, or casual expressions that break the intended tone.",
        "Assess whether the tone remains comprehensible (bureaucratic but not incoherent or overly opaque).",
        "Verify the tone does not distort meaning or introduce unintended sarcasm."
    ]
    tonality_metric = GEval(
        name="Tonality (Bureaucratese)",
        model=evaluator,
        evaluation_steps=tonality_steps,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )

    # 5) GEval: Safety（5 steps）
    safety_steps = [
        "Check for personal data such as emails, phone numbers, addresses, or uniquely identifying details not needed.",
        "Check for hateful, harassing, or toxic language; penalize any presence heavily.",
        "Verify there are no instructions enabling wrongdoing or unsafe behavior.",
        "Check for defamatory or ungrounded claims about real persons/organizations not supported by the input.",
        "Confirm the content remains professional and non-discriminatory."
    ]
    safety_metric = GEval(
        name="Safety",
        model=evaluator,
        evaluation_steps=safety_steps,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )

    # 6) execute measure()
    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    # 7) output key value
    return {
        "SummarizationScore": summarization_metric.score,
        "SummarizationReason": summarization_metric.reason,

        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": coherence_metric.reason,

        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,

        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason,
    }

In [56]:
results = evaluate_summary_with_deepeval(document_text, out.Summary, client)
print(results)

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.7272727272727273, 'SummarizationReason': 'The score is 0.73 because the summary introduces extra information that is not present in the original text, which may mislead the reader about the key points. Additionally, it fails to address a specific question that the original text can answer, indicating a lack of completeness in capturing the original message.', 'CoherenceScore': 0.8, 'CoherenceReason': 'The summary has a clear structure with an opening premise, key points about self-awareness and career management, and a conclusion emphasizing proactive planning. Ideas flow logically with effective transitions, though some sentences could be more concise. There is minimal ambiguity, but a few references could be clearer. Redundancy is low, and overall readability is good, with coherent paragraphs and appropriate language.', 'TonalityScore': 0.8, 'TonalityReason': "The response maintains a formal tone consistent with Bureaucratese, utilizing procedural phrasing an

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [67]:
def build_revision_prompt(
    context: document_text,
    summary_v1: str,
    eval_results: dict,
    tone: str = "Bureaucratese",
) -> list[dict[str, str]]:
    developer = {
        "role": "developer",
        "content": (
            "You are an expert editor. Revise the summary using ONLY the provided source text. "
            "Do not introduce any facts not present in the source. Preserve the requested tone."
        ),
    }

    user = {
        "role": "user",
        "content": f"""
TASK:
Revise the summary to improve quality based on the evaluation feedback.

REQUIREMENTS:
- Tone must remain: {tone}
- Do not add any new claims not supported by the source text.
- Improve coverage of key points (if missing), improve coherence/clarity, maintain concision.
- Output ONLY the revised summary text (no JSON).

SOURCE TEXT (context):
\"\"\"\n{context}\n\"\"\"

CURRENT SUMMARY (v1):
\"\"\"\n{summary_v1}\n\"\"\"

EVALUATION RESULTS (scores + reasons):
- SummarizationScore: {eval_results.get("SummarizationScore")}
- SummarizationReason: {eval_results.get("SummarizationReason")}

- CoherenceScore: {eval_results.get("CoherenceScore")}
- CoherenceReason: {eval_results.get("CoherenceReason")}

- TonalityScore: {eval_results.get("TonalityScore")}
- TonalityReason: {eval_results.get("TonalityReason")}

- SafetyScore: {eval_results.get("SafetyScore")}
- SafetyReason: {eval_results.get("SafetyReason")}

EDITING INSTRUCTIONS (apply all):
1) Fix any issues mentioned in the reasons above.
2) Ensure each paragraph has a single purpose and clear transitions.
3) Keep it succinct; remove redundancy.
4) Keep the Bureaucratese voice consistent (administrative, policy-like cadence).
""".strip()
    }

    return [developer, user]

In [68]:
def generate_revised_summary(client, messages, model="gpt-4o-mini", max_output_tokens=900, temperature=0):
    resp = client.responses.create(
        model=model,
        input=messages,
        max_output_tokens=max_output_tokens,
        temperature=temperature,
    )
    return resp.output_text

In [69]:
results_v1 = evaluate_summary_with_deepeval(document_text, out.Summary, client)
# Evaluate v1
summary_v1 = out.Summary
results_v1 = evaluate_summary_with_deepeval(document_text, summary_v1, client)

# Build revision prompt from context + v1 + evaluation
messages = build_revision_prompt(
    context=document_text,
    summary_v1=summary_v1,
    eval_results=results_v1,
    tone=out.Tone  # or "Bureaucratese"
)

# Generate v2
summary_v2 = generate_revised_summary(client, messages, model="gpt-4o-mini")

# Evaluate v2 using the same function
results_v2 = evaluate_summary_with_deepeval(document_text, summary_v2, client)

results_v1, summary_v2[:600], results_v2

Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

({'SummarizationScore': 0.6363636363636364,
  'SummarizationReason': 'The score is 0.64 because the summary contains contradictions to the original text, introduces extra information not present in the original, and leaves out key questions that the original text can answer, indicating a lack of fidelity to the source material.',
  'CoherenceScore': 0.8,
  'CoherenceReason': 'The summary has a clear structure with an opening premise, key points about self-awareness and career management, and a conclusion emphasizing proactive planning. Ideas flow logically with effective transitions, though some sentences could be more concise. There is minimal ambiguity, but a few references could be clearer. Redundancy is low, and overall readability is strong, with coherent paragraphs and appropriate language.',
  'TonalityScore': 0.7,
  'TonalityReason': "The response maintains a formal tone and includes structured enumerations, aligning well with Bureaucratese. However, it contains some phrases th

In [70]:
# Building matrics with deltas and overall better/worse flag
def compare_eval(v1: dict, v2: dict) -> dict:
    metrics = ["SummarizationScore", "CoherenceScore", "TonalityScore", "SafetyScore"]

    deltas = {
        m.replace("Score", "Delta"): (v2.get(m, 0) or 0) - (v1.get(m, 0) or 0)
        for m in metrics
    }

    better = sum(d > 0 for d in deltas.values()) >= 2

    return {
        "Before": v1,
        "After": v2,
        "Deltas": deltas,
        "BetterOutput": better,
    }

In [72]:
final_report = compare_eval(results_v1, results_v2)
final_report

{'Before': {'SummarizationScore': 0.6363636363636364,
  'SummarizationReason': 'The score is 0.64 because the summary contains contradictions to the original text, introduces extra information not present in the original, and leaves out key questions that the original text can answer, indicating a lack of fidelity to the source material.',
  'CoherenceScore': 0.8,
  'CoherenceReason': 'The summary has a clear structure with an opening premise, key points about self-awareness and career management, and a conclusion emphasizing proactive planning. Ideas flow logically with effective transitions, though some sentences could be more concise. There is minimal ambiguity, but a few references could be clearer. Redundancy is low, and overall readability is strong, with coherent paragraphs and appropriate language.',
  'TonalityScore': 0.7,
  'TonalityReason': "The response maintains a formal tone and includes structured enumerations, aligning well with Bureaucratese. However, it contains some 

Please, do not forget to add your comments.

1) Did you get a better output? Why?

In expectation, the output is usually better, particularly in Summarization and Coherence.
This is because the v2 prompt converts the evaluation reasons into explicit revision constraints through filling missing key points, reorganizing structure, removing redundancy. However, the revised summary (v2) did not produce a better overall outcome.
The most significant change was in the SummarizationScore, which dropped substantially from 0.86 to 0.53 (Δ = -0.32). According to the evaluation reason, the revised version introduced claims that were not explicitly supported by the original text, which indicates potential factual drift or mild hallucination and still failed to address the specific missing element identified in v1.

The other dimensions remained unchanged (CoherenceScore, TonalityScore,SafetyScore stayed at 0.80.)
This indicates that the revision loop preserved structure, readability, tone consistency, and safety
Overall,the system did not significantly improve the output. This might be caused the reason that the evaluation feedback in v1 identified one key deficiency of incomplete coverage of the strengths versus weaknesses principle. However, the revision prompt likely encouraged global rewriting rather than targeted correction.

2) Are these controls enough?

No, the current control loop is insufficient. Although coherence, tone, and safety remained stable, summarization dropped. This indicates sensitivity of the current control loop to subtle content variations and evaluation variance, which does not automatically guarantee improvement.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
